# Neuron Forge — 250M from-scratch training

This notebook trains the repository's **248,144,832-parameter** decoder-only Transformer from random initialization. It does not download pretrained model weights or call an AI API.

## Before running
1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU if one is offered).
2. Run the cells in order. The notebook checks for a real CUDA GPU and stops if none is attached.
3. Sign in to Google Drive when prompted. Checkpoints and the corpus are stored there so a later session can resume.

Free GPU availability and session length are controlled by Colab and are not guaranteed. This run is resumable; if the session ends, reopen this notebook and run the training cell again. The current target is 100,000 optimizer steps, which may require many sessions.

**Honest limitation:** a 250M parameter model trained on a limited Wikipedia corpus will be an early research model, not a GPT-6-level assistant. It needs much more varied data, instruction tuning, and evaluation before it can be used as a dependable chat model.

Training text source: English Wikipedia dump subset `20231101.en` via `wikimedia/wikipedia`. The dataset card identifies the source text as CC BY-SA 3.0 and GFDL; retain the attribution record saved alongside the corpus.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO = Path('/content/Neuron-Forge')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/hellothiseasytouse-crypto/Neuron-Forge.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers>=0.21,<1', 'datasets>=3.0', 'pyyaml>=6.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
print('Repository ready:', REPO)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. In Colab choose Runtime → Change runtime type → GPU, then rerun this cell.')
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / (1024 ** 3)
print('GPU:', gpu.name)
print(f'GPU memory: {gpu_gib:.1f} GiB')
print('CUDA:', torch.version.cuda)
print('BF16 supported:', torch.cuda.is_bf16_supported())
if gpu_gib < 12:
    raise RuntimeError('This 250M configuration needs a GPU with at least 12 GiB of memory for this training setup.')
print('GPU check passed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/NeuronForge250M')
RUN_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_PATH = RUN_DIR / 'data' / 'general_corpus.txt'
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / 'latest.pt'
(RUN_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
print('Persistent run directory:', RUN_DIR)
print('Existing resumable checkpoint:', CHECKPOINT_PATH.exists())

In [ ]:
# Build a repeatable general-text corpus from English Wikipedia.
# The existing corpus is reused on later sessions; it is not downloaded again.
from datasets import load_dataset

TARGET_WORDS = 12_000_000
MAX_CHARS = 100_000_000
if CORPUS_PATH.exists() and len(CORPUS_PATH.read_text(encoding='utf-8').split()) >= 10_000_000:
    print('Using existing corpus:', CORPUS_PATH)
else:
    CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    stream = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
    words = 0
    chars = 0
    with CORPUS_PATH.open('w', encoding='utf-8') as out:
        for article in stream:
            body = (article.get('text') or '').strip()
            if len(body) < 300:
                continue
            title = (article.get('title') or 'Untitled').strip()
            url = (article.get('url') or '').strip()
            record = f'\n\n# {title}\nSource: {url}\n{body}\n'
            remaining = MAX_CHARS - chars
            if remaining <= 0:
                break
            record = record[:remaining]
            out.write(record)
            chars += len(record)
            words += len(body.split())
            if words >= TARGET_WORDS or chars >= MAX_CHARS:
                break
    text = CORPUS_PATH.read_text(encoding='utf-8')
    words = len(text.split())
    if words < 10_000_000:
        raise RuntimeError(f'Corpus too small: {words:,} words. Need at least 10,000,000 words; do not start training yet.')
    attribution = (
        'Neuron Forge training corpus source: Wikimedia Foundation, English Wikipedia dump subset 20231101.en.\\n'
        'Dataset: https://huggingface.co/datasets/wikimedia/wikipedia\\n'
        'Source terms: English Wikipedia text is available under CC BY-SA 3.0 and the GNU Free Documentation License (GFDL).\\n'
        'Wikimedia dump legal information: https://dumps.wikimedia.org/legal.html\\n'
        'Article titles and source URLs are included in the corpus for attribution. Review and comply with source license terms before redistributing data, weights, or outputs.'
    )
    (RUN_DIR / 'CORPUS_ATTRIBUTION.txt').write_text(attribution, encoding='utf-8')
    print(f'Corpus prepared: {words:,} words, {chars:,} characters')
    print('Saved:', CORPUS_PATH)

print(f'Corpus size: {CORPUS_PATH.stat().st_size / (1024**2):.1f} MiB')

In [ ]:
from neuronforge_engine.config import load_config
from neuronforge_engine.training import train

cfg = load_config('configs/250m.yaml')
cfg.train.data_path = str(CORPUS_PATH)
cfg.train.checkpoint_path = str(CHECKPOINT_PATH)
cfg.train.device = 'cuda'
cfg.train.max_steps = 100_000  # total target; repeated sessions resume at the saved optimizer step
cfg.train.batch_size = 1
cfg.train.gradient_accumulation_steps = 8
cfg.train.gradient_checkpointing = True
cfg.train.eval_interval = 250  # checkpoint with optimizer state every 250 steps
cfg.train.eval_batches = 2
cfg.train.warmup_steps = 2_000

expected = 248_144_832
assert cfg.model.parameter_count() == expected, cfg.model.parameter_count()
print(f'Model: {expected:,} parameters | {cfg.model.n_layer} layers | {cfg.model.n_head} heads | width {cfg.model.n_embd}')
print('Resume:', CHECKPOINT_PATH.exists())
train(cfg, resume=CHECKPOINT_PATH.exists())

In [ ]:
# Run this cell after training finishes (or after a session resumes and finishes).
from neuronforge_engine.checkpoint import read_checkpoint
from neuronforge_engine.config import load_config

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError('No checkpoint found. Run the training cell first.')
latest = read_checkpoint(CHECKPOINT_PATH, device='cpu')
best_path = CHECKPOINT_PATH.with_name('best.pt')
best = read_checkpoint(best_path, device='cpu') if best_path.exists() else latest
status = {
    'parameters': sum(t.numel() for t in latest['model_state'].values()),
    'optimizer_step': latest['step'],
    'best_checkpoint_step': best['step'],
    'best_validation_loss': best['best_val_loss'],
    'checkpoint_format': latest['format_version'],
    'checkpoint_path': str(CHECKPOINT_PATH),
}
print(status)
print('This is a trained checkpoint only after optimizer_step is greater than zero.')